In [15]:
from pathlib import Path
import re
import pandas as pd
import pdfplumber

# =========================
# CONFIG
# =========================
PDF_PATH = Path(r"/home/hello/Desktop/Documents/Case_vs_BBG/Bloomberg´s evidence/(2) Leonardo Fabian Maranon Mejia v Bloomberg LP - Disclosure bundle of documents 11.12.2024_SEARCHABLE.pdf")
PAGE_START, PAGE_END = 220, 238  # 1-indexed page numbers
OUT_DIR = Path("./extract_out_utc_v3_1")
OUT_DIR.mkdir(exist_ok=True)

# =========================
# REGEX
# =========================
DATE_DMY_RE = re.compile(r"(\d{1,2}/\d{1,2}/\d{4})")           # dd/mm/yyyy (1-digit allowed)
TIME_RE = re.compile(r"\b(\d{2}:\d{2}(?::\d{2})?)\b")          # HH:MM or HH:MM:SS
MY_ANYWHERE_RE = re.compile(r"(\d{1,2})/(\d{4})")              # m/yyyy anywhere (even inside tokens)

F_TOKEN_RE = re.compile(r"\b(F\d{2,8}\d{1,2}/\d{4})\b|\b(F\d{2,8})\b")

KEYWORDS = ["LPAD", "Launchpad", "WorkCentre", "WorkCenter", "crash", "crashed", "crashes", "queue", "ticket", "ADD"]

# =========================
# HELPERS
# =========================
def norm_cell(x) -> str:
    if x is None:
        return ""
    s = str(x)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def extract_month_year_anywhere(s: str):
    """
    Finds any m/yyyy occurrence inside the string (even inside tokens),
    keeps only valid months 1..12, returns last valid occurrence as MM/YYYY.
    """
    if not s:
        return None
    matches = MY_ANYWHERE_RE.findall(s)
    valid = [(m, y) for (m, y) in matches if 1 <= int(m) <= 12]
    if not valid:
        return None
    m, y = valid[-1]
    return f"{int(m):02d}/{y}"

def extract_full_date(s: str):
    m = DATE_DMY_RE.search(s or "")
    return m.group(1) if m else None

def extract_time(s: str):
    m = TIME_RE.search(s or "")
    return m.group(1) if m else None

def extract_f_token(s: str):
    if not s:
        return None
    m = F_TOKEN_RE.search(s)
    if not m:
        return None
    return m.group(1) or m.group(2)

# =========================
# EXTRACTION
# =========================
rows = []

with pdfplumber.open(PDF_PATH) as pdf:
    for pno in range(PAGE_START, PAGE_END + 1):
        page = pdf.pages[pno - 1]

        # ---- 1) Try tables first ----
        tables = page.extract_tables() or []
        if tables:
            for ti, table in enumerate(tables):
                for ri, r in enumerate(table):
                    cells = [norm_cell(c) for c in r]
                    joined = " | ".join([c for c in cells if c]).strip()
                    if not joined:
                        continue

                    is_relevant = any(k.lower() in joined.lower() for k in KEYWORDS)
                    date_utc = extract_full_date(joined)
                    month_year_utc = extract_month_year_anywhere(joined) if not date_utc else None
                    time_utc = extract_time(joined)
                    f_token = extract_f_token(joined)

                    rows.append({
                        "page": pno,
                        "source": f"table[{ti}]",
                        "row_idx": ri,
                        "date_utc": date_utc,
                        "month_year_utc": month_year_utc,
                        "time_utc": time_utc,
                        "f_token": f_token,
                        "relevant_flag": is_relevant,
                        "row_text": joined
                    })
            continue  # don't double-count with fallback if table extraction worked

        # ---- 2) Fallback: reconstruct lines from words ----
        words = page.extract_words() or []
        if not words:
            continue

        tol = 3  # line grouping tolerance
        words_sorted = sorted(words, key=lambda w: (round(w["top"] / tol) * tol, w["x0"]))
        current_key = None
        current_words = []

        def flush_line():
            if not current_words:
                return

            line = " ".join(w["text"] for w in current_words)
            line = re.sub(r"\s+", " ", line).strip()

            # clear in-place (no rebinding => no nonlocal needed)
            current_words.clear()

            if not line:
                return

            is_relevant = any(k.lower() in line.lower() for k in KEYWORDS)
            date_utc = extract_full_date(line)
            month_year_utc = extract_month_year_anywhere(line) if not date_utc else None
            time_utc = extract_time(line)
            f_token = extract_f_token(line)

            rows.append({
                "page": pno,
                "source": "words_line",
                "row_idx": None,
                "date_utc": date_utc,
                "month_year_utc": month_year_utc,
                "time_utc": time_utc,
                "f_token": f_token,
                "relevant_flag": is_relevant,
                "row_text": line
            })

        for w in words_sorted:
            key = round(w["top"] / tol) * tol
            if current_key is None:
                current_key = key
            if key != current_key:
                flush_line()
                current_key = key
            current_words.append(w)

        flush_line()

df = pd.DataFrame(rows)

# =========================
# OUTPUTS (Pure extraction)
# =========================
# =========================
# Build df_ts
# =========================
df_ts = df[
    df["time_utc"].notna() &
    (df["date_utc"].notna() | df["month_year_utc"].notna())
].copy()

df_ts["date_like"] = df_ts["date_utc"].fillna(df_ts["month_year_utc"])
df_ts["timestamp_utc_str"] = df_ts["date_like"] + " " + df_ts["time_utc"]

df_ts["dt_parse"] = pd.to_datetime(
    df_ts["timestamp_utc_str"],
    errors="coerce",
    dayfirst=True
)

# =========================
# Timezone enrichment (ONLY where full date exists)
# =========================
import pandas as pd
from zoneinfo import ZoneInfo

def add_pit_and_time_category(
    df_ts: pd.DataFrame,
    dt_parse_col: str = "dt_parse",
    uk_tz: str = "Europe/London",
) -> pd.DataFrame:
    """
    Enrich df_ts with:
      - PIT timestamps (UTC + UK local, DST-aware)
      - UK offset + DST flags
      - UK hour/minute + weekday/weekend
      - One clean classification column: time_category ∈ {WEEKEND, LUNCH, WORK, OUTSIDE}

    Assumptions:
      - df_ts[dt_parse_col] is a naive datetime representing a UTC clock reading.
      - Rows where dt_parse is NaT remain un-enriched (kept as NaN/NA).
    """
    out = df_ts.copy()
    UK_TZ = ZoneInfo(uk_tz)

    mask = out[dt_parse_col].notna()

    # ---- PIT (UTC) ----
    out.loc[mask, "dt_utc"] = out.loc[mask, dt_parse_col].dt.tz_localize("UTC")

    # ---- PIT (UK local) ----
    out.loc[mask, "dt_uk"] = out.loc[mask, "dt_utc"].dt.tz_convert(UK_TZ)

    # ---- Offsets / DST ----
    out.loc[mask, "uk_offset"] = out.loc[mask, "dt_uk"] - out.loc[mask, "dt_utc"]
    out.loc[mask, "uk_offset_minutes"] = (
        out.loc[mask, "uk_offset"].dt.total_seconds().div(60).astype("Int64")
    )
    out.loc[mask, "uk_offset_hours"] = out.loc[mask, "uk_offset_minutes"] / 60
    out.loc[mask, "uk_tz_abbr"] = out.loc[mask, "dt_uk"].dt.strftime("%Z")  # GMT or BST
    out.loc[mask, "uk_is_dst"] = out.loc[mask, "uk_offset_hours"] == 1

    # ---- Readable PIT strings ----
    out.loc[mask, "pit_utc_iso"] = out.loc[mask, "dt_utc"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    out.loc[mask, "pit_uk_iso"] = out.loc[mask, "dt_uk"].dt.strftime("%Y-%m-%dT%H:%M:%S%z")
    out.loc[mask, "pit"] = out.loc[mask, "dt_uk"].dt.strftime("%Y-%m-%d %H:%M:%S %Z")  # watch-time

    # ---- Time components (UK local) ----
    out.loc[mask, "uk_hour"] = out.loc[mask, "dt_uk"].dt.hour
    out.loc[mask, "uk_minute"] = out.loc[mask, "dt_uk"].dt.minute
    out.loc[mask, "uk_weekday"] = out.loc[mask, "dt_uk"].dt.weekday  # 0=Mon ... 6=Sun
    out.loc[mask, "is_weekend"] = out.loc[mask, "uk_weekday"] >= 5

    # ---- Single clean classification column ----
    # Priority: WEEKEND > LUNCH > WORK > OUTSIDE
    out["time_category"] = pd.NA
    out.loc[mask, "time_category"] = "OUTSIDE"
    out.loc[mask & (out["is_weekend"] == True), "time_category"] = "WEEKEND"
    out.loc[mask & (out["is_weekend"] == False) & (out["uk_hour"] >= 12) & (out["uk_hour"] < 13), "time_category"] = "LUNCH"
    out.loc[mask & (out["is_weekend"] == False) & (out["uk_hour"] >= 8) & (out["uk_hour"] < 18) & (out["time_category"] != "LUNCH"), "time_category"] = "WORK"

    return out

# Usage:
df_ts = add_pit_and_time_category(df_ts)


# =========================
# Save AFTER enrichment
# =========================
df.to_csv(OUT_DIR / "raw_rows_p220_238.csv", index=False)

df_ts.sort_values(["dt_parse", "page"], na_position="last") \
     .to_csv(OUT_DIR / "timestamps_p220_238.csv", index=False)



print("=== v3.1 extraction summary ===")
print(f"PDF: {PDF_PATH.name}")
print(f"Pages: {PAGE_START}-{PAGE_END}")
print(f"Raw rows captured: {len(df)}")
print(f"Timestamp-like rows (time + date_or_monthyear): {len(df_ts)}")
print("Saved:")
print(" -", (OUT_DIR / "raw_rows_p220_238.csv").resolve())
print(" -", (OUT_DIR / "timestamps_p220_238.csv").resolve())


=== v3.1 extraction summary ===
PDF: (2) Leonardo Fabian Maranon Mejia v Bloomberg LP - Disclosure bundle of documents 11.12.2024_SEARCHABLE.pdf
Pages: 220-238
Raw rows captured: 1088
Timestamp-like rows (time + date_or_monthyear): 886
Saved:
 - /home/hello/Projects/Statements/bbg_specific_bundle/extract_out_utc_v3_1/raw_rows_p220_238.csv
 - /home/hello/Projects/Statements/bbg_specific_bundle/extract_out_utc_v3_1/timestamps_p220_238.csv


/tmp/ipykernel_221631/3767889680.py:171: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_ts["dt_parse"] = pd.to_datetime(


In [17]:
df_ts.groupby("time_category").size()

time_category
LUNCH       82
OUTSIDE      7
WEEKEND    194
WORK       602
dtype: int64

In [13]:
from pathlib import Path
import re
import pandas as pd
import pdfplumber

# =========================================================
# CONFIG
# =========================================================
PDF_PATH = Path(
    "/home/hello/Desktop/Documents/Case_vs_BBG/Bloomberg´s evidence/"
    "(2) Leonardo Fabian Maranon Mejia v Bloomberg LP - Disclosure bundle of documents 11.12.2024_SEARCHABLE.pdf"
)

# Scan a BROADER window first.
# These are 1-indexed PDF display pages in pdfplumber terms (page number = index + 1).
PDF_PAGE_START = 238
PDF_PAGE_END = 255

OUT_DIR = Path("./badge_weekend_extract_v2")
OUT_DIR.mkdir(exist_ok=True)

TARGET_NAME = "LEONARDO FABIAN MARANON MEJIA"

# =========================================================
# REGEX
# =========================================================
WEEKEND_DT_RE = re.compile(
    r"(?P<date>"
    r"(?:Saturday|Sunday),?\s+"
    r"(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+"
    r"\d{1,2},?\s+2023"
    r")\s+"
    r"(?P<time>\d{1,2}:\d{2}(?::\d{2})?\s*(?:AM|PM))",
    flags=re.I
)

UUID_RE = re.compile(r"\b(\d{6,10})\b")

# =========================================================
# HELPERS
# =========================================================
def norm_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = s.replace("‘", "").replace("’", "").replace("`", "").replace("´", "")
    s = s.replace("“", '"').replace("”", '"')
    s = re.sub(r"\s+", " ", s).strip()
    return s


def page_text_lines(page) -> list[str]:
    """
    Extract page text and split into normalized lines.
    For this badge section, extract_text() appears to preserve rows better
    than rebuilding from word coordinates.
    """
    text = page.extract_text()
    if not text:
        return []

    lines = []
    for line in text.splitlines():
        line = norm_text(line)
        if line:
            lines.append(line)
    return lines


def extract_candidate_lines(pdf_path: Path, page_start: int, page_end: int) -> pd.DataFrame:
    """
    Broad scan for candidate badge rows.
    Keeps:
      - raw line
      - page number
      - whether line contains target name
      - whether line contains weekend datetime pattern
    """
    rows = []

    with pdfplumber.open(pdf_path) as pdf:
        total_pages = len(pdf.pages)
        print(f"Total PDF pages detected: {total_pages}")

        start_idx = max(page_start - 1, 0)
        end_idx = min(page_end - 1, total_pages - 1)

        for idx in range(start_idx, end_idx + 1):
            page = pdf.pages[idx]
            lines = page_text_lines(page)

            for line_no, line in enumerate(lines):
                has_name = TARGET_NAME in line.upper()
                has_weekend_dt = bool(WEEKEND_DT_RE.search(line))

                if has_name or has_weekend_dt:
                    rows.append({
                        "pdf_page_index": idx,
                        "display_page": idx + 1,
                        "line_no": line_no,
                        "has_name": has_name,
                        "has_weekend_dt": has_weekend_dt,
                        "raw_line": line,
                    })

    return pd.DataFrame(rows)


def parse_candidate_line(line: str) -> dict | None:
    """
    Parse only the fields we actually need for hours.
    """
    raw = norm_text(line)

    if TARGET_NAME not in raw.upper():
        return None

    m_dt = WEEKEND_DT_RE.search(raw)
    if not m_dt:
        return None

    m_uuid = UUID_RE.search(raw)
    uuid = m_uuid.group(1) if m_uuid else None

    date_str = norm_text(m_dt.group("date"))
    time_str = norm_text(m_dt.group("time")).upper().replace(" ", "")

    dt = pd.to_datetime(f"{date_str} {time_str}", errors="coerce")
    if pd.isna(dt):
        dt = pd.to_datetime(f"{date_str.replace(',', '')} {time_str}", errors="coerce")

    if pd.isna(dt):
        return None

    return {
        "UUID": uuid,
        "Name": TARGET_NAME,
        "Date": dt.date(),
        "Time": dt.time(),
        "DateTime": dt,
        "weekday": dt.day_name(),
        "is_weekend": dt.weekday() >= 5,
        "raw_line": raw,
    }


def build_badge_df(candidate_df: pd.DataFrame) -> pd.DataFrame:
    parsed = []

    if candidate_df.empty:
        return pd.DataFrame(columns=[
            "UUID", "Name", "Date", "Time", "DateTime",
            "weekday", "is_weekend", "raw_line",
            "pdf_page_index", "display_page", "line_no"
        ])

    for rec in candidate_df.to_dict("records"):
        row = parse_candidate_line(rec["raw_line"])
        if row is not None:
            row["pdf_page_index"] = rec["pdf_page_index"]
            row["display_page"] = rec["display_page"]
            row["line_no"] = rec["line_no"]
            parsed.append(row)

    badge_df = pd.DataFrame(parsed)

    if badge_df.empty:
        return pd.DataFrame(columns=[
            "UUID", "Name", "Date", "Time", "DateTime",
            "weekday", "is_weekend", "raw_line",
            "pdf_page_index", "display_page", "line_no"
        ])

    badge_df = (
        badge_df
        .drop_duplicates(subset=["DateTime", "raw_line"])
        .sort_values(["DateTime", "display_page", "line_no"])
        .reset_index(drop=True)
    )

    return badge_df


def infer_in_out_by_date(badge_df: pd.DataFrame) -> pd.DataFrame:
    """
    OCR makes IN/OUT unreliable.
    Infer sequence within each day:
      0 -> IN
      1 -> OUT
      2 -> IN
      3 -> OUT
    """
    if badge_df.empty:
        out = badge_df.copy()
        out["seq_in_day"] = pd.Series(dtype="Int64")
        out["IN/OUT"] = pd.Series(dtype="object")
        return out

    out = badge_df.copy().sort_values("DateTime").reset_index(drop=True)
    out["seq_in_day"] = out.groupby("Date").cumcount()
    out["IN/OUT"] = out["seq_in_day"].apply(lambda x: "IN" if x % 2 == 0 else "OUT")
    return out


def build_date_audit(df_inferred: pd.DataFrame) -> pd.DataFrame:
    if df_inferred.empty:
        return pd.DataFrame(columns=["Date", "n_rows", "odd_flag"])

    audit = (
        df_inferred.groupby("Date")
        .size()
        .reset_index(name="n_rows")
        .sort_values("Date")
        .reset_index(drop=True)
    )
    audit["odd_flag"] = audit["n_rows"] % 2 != 0
    return audit


def compute_sessions(df_inferred: pd.DataFrame) -> pd.DataFrame:
    sessions = []

    if df_inferred.empty:
        return pd.DataFrame(columns=[
            "date", "weekday", "in_time", "out_time", "hours", "n_rows_in_date"
        ])

    for dt_date, g in df_inferred.groupby("Date", sort=True):
        g = g.sort_values("DateTime").reset_index(drop=True)
        n = len(g)

        for i in range(0, n - 1, 2):
            row_in = g.iloc[i]
            row_out = g.iloc[i + 1]

            if row_out["DateTime"] >= row_in["DateTime"]:
                hours = (row_out["DateTime"] - row_in["DateTime"]).total_seconds() / 3600.0
                sessions.append({
                    "date": dt_date,
                    "weekday": row_in["weekday"],
                    "in_time": row_in["DateTime"],
                    "out_time": row_out["DateTime"],
                    "hours": round(hours, 2),
                    "n_rows_in_date": n,
                })

    return pd.DataFrame(sessions)


def build_daily_hours(sessions_df: pd.DataFrame) -> pd.DataFrame:
    if sessions_df.empty:
        return pd.DataFrame(columns=["date", "weekday", "hours"])

    return (
        sessions_df.groupby(["date", "weekday"], as_index=False)["hours"]
        .sum()
        .sort_values("date")
        .reset_index(drop=True)
    )


# =========================================================
# RUN
# =========================================================
candidate_df = extract_candidate_lines(PDF_PATH, PDF_PAGE_START, PDF_PAGE_END)
badge_df = build_badge_df(candidate_df)
badge_df_inferred = infer_in_out_by_date(badge_df)
date_audit_df = build_date_audit(badge_df_inferred)
sessions_df = compute_sessions(badge_df_inferred)
daily_df = build_daily_hours(sessions_df)

total_hours = round(daily_df["hours"].sum(), 2) if not daily_df.empty else 0.0

# =========================================================
# PRINTS
# =========================================================
print("\n=== CANDIDATE LINES ===")
print(len(candidate_df))

print("\n=== PARSED BADGE ROWS ===")
print(len(badge_df))

print("\n=== BADGE DF PREVIEW ===")
print(badge_df.head(25).to_string(index=False))

print("\n=== DATE AUDIT ===")
print(date_audit_df.to_string(index=False))

print("\n=== ODD-COUNT DATES ===")
odd_df = date_audit_df[date_audit_df["odd_flag"] == True]
if odd_df.empty:
    print("None")
else:
    print(odd_df.to_string(index=False))

print("\n=== SESSIONS ===")
print(sessions_df.to_string(index=False))

print("\n=== DAILY HOURS ===")
print(daily_df.to_string(index=False))

print(f"\n=== TOTAL WEEKEND HOURS ===\n{total_hours}")

# =========================================================
# SAVE
# =========================================================
candidate_csv = OUT_DIR / "candidate_lines.csv"
badge_csv = OUT_DIR / "badge_df.csv"
badge_inferred_csv = OUT_DIR / "badge_df_inferred.csv"
audit_csv = OUT_DIR / "date_audit.csv"
sessions_csv = OUT_DIR / "sessions.csv"
daily_csv = OUT_DIR / "daily_hours.csv"

candidate_df.to_csv(candidate_csv, index=False)
badge_df.to_csv(badge_csv, index=False)
badge_df_inferred.to_csv(badge_inferred_csv, index=False)
date_audit_df.to_csv(audit_csv, index=False)
sessions_df.to_csv(sessions_csv, index=False)
daily_df.to_csv(daily_csv, index=False)

print("\n=== SAVED FILES ===")
print(candidate_csv.resolve())
print(badge_csv.resolve())
print(badge_inferred_csv.resolve())
print(audit_csv.resolve())
print(sessions_csv.resolve())
print(daily_csv.resolve())

Total PDF pages detected: 560

=== CANDIDATE LINES ===
158

=== PARSED BADGE ROWS ===
80

=== BADGE DF PREVIEW ===
   UUID                          Name       Date     Time            DateTime  weekday  is_weekend                                                                                                                                                                                     raw_line  pdf_page_index  display_page  line_no
1877712 LEONARDO FABIAN MARANON MEJIA 2023-01-07 11:03:00 2023-01-07 11:03:00 Saturday        True   1877712 LEONARDO FABIAN MARANON MEJIA Saturday, January 7, 2023 11:03:00AM Bloomberg Financial Solutions London -3 Queen Victoria Street London United Kingdom United Kingdom London - avs             240           241       13
1877712 LEONARDO FABIAN MARANON MEJIA 2023-01-07 16:46:00 2023-01-07 16:46:00 Saturday        True    1877712 LEONARDO FABIAN MARANON MEJIA Saturday, January 7, 2023 4:46:00PM Bloomberg Financial Solutions London -3 Queen Victoria 